# v41 — Mac mini M4 / MPS
После AirDrop всей папки запусти **Start_v41_Mac.command**, выбери kernel **Car ReID Mac M4**, нажми **Run All**.
Три очереди по очереди на Apple GPU: длинный TransReID, NiVe BN/alpha и DINOv2/R50-IBN. До 55 конфигураций, без ограничения времени.
Данные уже находятся в research_transfer/v41_inputs. ZIP повторно распаковывать не нужно. Старые перенесённые .venv не использовать.
Настоящий P×K, float32, без CPU fallback и скрытого изменения batch. Контроль извлекается на этом Mac заново.
MVP_fusion_v25, bbox, validation и исторические runs не изменяются. Остановка/продолжение: тот же RUN_NAME и то же окружение на этом Mac.
NiVe1303: https://data.mendeley.com/datasets/42wv2svztx/1 — нужный train уже перенесён вместе с папкой.


In [ ]:
import os, sys, subprocess
from pathlib import Path
if 'torch' in sys.modules and (os.environ.get('PYTORCH_ENABLE_MPS_FALLBACK') != '0' or os.environ.get('PYTORCH_MPS_FAST_MATH') != '0'):
    raise RuntimeError('Restart Kernel: настройки MPS нужно задать до импорта torch')
os.environ.update(ORT_DISABLE_TELEMETRY='1', OPENBLAS_NUM_THREADS='1', OMP_NUM_THREADS='4', PYTORCH_ENABLE_MPS_FALLBACK='0', PYTORCH_MPS_FAST_MATH='0', XFORMERS_DISABLED='1')
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'training/research_gpu.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Открой notebook внутри перенесённой папки Car-classification-MSK')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if Path(sys.prefix).resolve() != (ROOT / '.venv-v41-m4').resolve():
    raise RuntimeError('Выбери Car ReID Mac M4 после Start_v41_Mac.command. Перенесённая .venv не подходит.')
INPUTS = ROOT / 'research_transfer/v41_inputs'
if not (INPUTS / 'inputs.json').is_file():
    raise FileNotFoundError('Дождись полного переноса папки, включая research_transfer/v41_inputs')
RUN_NAME = 'mac_m4_v1'
print('Python:', sys.executable, '\nRepo:', ROOT, '\nInputs:', INPUTS, '\nRun:', RUN_NAME)


In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_parallel_research.py', 'tests/test_research_mac.py'], cwd=ROOT, env=os.environ.copy(), check=True)
subprocess.run([sys.executable, '-m', 'training.research_mac'], cwd=ROOT, env=os.environ.copy(), check=True)


In [ ]:
from training.research_gpu import run
summary = run(INPUTS, RUN_NAME, device='mps')


In [ ]:
print('Статус:', summary['status'])
for item in summary['leaderboard'][:10]:
    print(item['trial'], 'rung', item['rung'], 'G2:', round(item['means']['G2'],6), 'delta:', round(item['delta'],6))
print('Сохрани notebook и передай обратно ОБА архива:')
for suffix in ('analysis', 'full_results'):
    print(ROOT / f'OSNet-AIN-x1.0/variant_41_windows_queues/{RUN_NAME}_{suffix}.zip')
